# Devoir Week 2 — Pipeline NLP pour la structuration du Code de la Route

**Master IASD 2026 | Pr. I. BENABDELOUAHAB**  
Université Abdelmalek Essaadi — FST Tanger

---

## Objectif
Construire un pipeline NLP complet capable de transformer un texte juridique brut (arabe **et** français) du Code de la Route Marocain en un fichier CSV structuré `export_final.csv`.


**Réalisée par ACHARF EL BOUMASHOULI**

**Master IASD 2026 | Pr. I. BENABDELOUAHAB**


## 1. Importation des bibliothèques

In [26]:
import re
import json
import unicodedata
import pandas as pd
import numpy as np
from collections import Counter

try:
    import pyarabic.araby as araby
    from pyarabic.araby import strip_tashkeel, strip_tatweel, normalize_hamza
    PYARABIC_AVAILABLE = True
    print("✔ PyArabic chargé avec succès")
except ImportError:
    PYARABIC_AVAILABLE = False
    print("⚠ PyArabic non disponible — normalisation manuelle utilisée")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import LabelEncoder
print("✔ scikit-learn chargé")

try:
    import spacy
    # Essayer de charger le modèle français
    try:
        nlp_fr = spacy.load("fr_core_news_sm")
        SPACY_AVAILABLE = True
        print("✔ spaCy modèle français chargé")
    except OSError:
        # Créer un pipeline vide comme fallback
        nlp_fr = spacy.blank("fr")
        SPACY_AVAILABLE = False
        print("⚠ Modèle spaCy fr non installé — utilisation du NER custom")
except ImportError:
    SPACY_AVAILABLE = False
    print("⚠ spaCy non disponible")

print("\n✅ Toutes les bibliothèques chargées.")

✔ PyArabic chargé avec succès
✔ scikit-learn chargé
✔ spaCy modèle français chargé

✅ Toutes les bibliothèques chargées.


---
## 2. Données brutes — Corpus Code de la Route Marocain

Ce corpus est extrait du **Bulletin Officiel n° 5874** (16-9-2010) contenant le Code de la Route Marocain.  
Source : https://www.sgg.gov.ma/BulletinOfficiel.aspx

**Version arabe + version française** incluses conformément aux consignes.

In [27]:
# ==============================================================
# CORPUS ARABE — extraits représentatifs du Code de la Route
# ==============================================================
raw_text_arabic = """
مادة 161
يُحدَّد الحد الأقصى للسرعة داخل التجمعات العمرانية في 60 كيلومترا في الساعة لجميع المركبات.
يعاقب كل سائق تجاوز هذه السرعة بغرامة 700 درهم مع خصم 4 نقط من رخصة السياقة.

مادة 162
يُحدَّد الحد الأقصى للسرعة خارج التجمعات العمرانية في 100 كيلومترا في الساعة للسيارات الخفيفة.
يعاقب كل سائق تجاوز هذه السرعة بغرامة 700 درهم مع خصم 4 نقط.

مادة 163
يُحدَّد الحد الأقصى للسرعة على الطريق السيار في 120 كيلومترا في الساعة.
في حالة تجاوز هذه السرعة، يعاقب السائق بغرامة 1500 درهم مع خصم 6 نقط من رخصة السياقة.

مادة 164
يُمنع الوقوف في الأماكن الآتية: أمام المستشفيات، أمام المدارس، على الجسور، في النفق.
يعاقب على مخالفة هذه المقتضيات بغرامة 400 درهم مع خصم 2 نقطة من رخصة السياقة.

مادة 165
يجب على السائق ربط حزام الأمان طوال مدة السياقة داخل وخارج التجمعات العمرانية.
يعاقب على مخالفة هذا الالتزام بغرامة 300 درهم مع خصم 2 نقطة.

مادة 166
يُمنع استعمال الهاتف النقال أثناء قيادة المركبة دون استعمال جهاز لاسلكي حر اليدين.
يعاقب على ذلك بغرامة 500 درهم مع خصم 2 نقطة من رخصة السياقة.

مادة 167
يُلزم سائقو الدراجات النارية وركابها بارتداء خوذة واقية معتمدة.
يعاقب على مخالفة هذا الإلزام بغرامة 300 درهم مع خصم 2 نقطة.

مادة 168
يُحدَّد الحد الأقصى للسرعة لمركبات نقل البضائع الثقيلة خارج التجمعات العمرانية في 80 كيلومترا في الساعة.
يعاقب كل سائق شاحنة تجاوز هذه السرعة بغرامة 700 درهم مع خصم 4 نقط.

مادة 169
يُلزم كل سائق بالتوقف عند إشارة التوقف (STOP) وإعطاء الأولوية.
يعاقب على عدم الامتثال بغرامة 500 درهم مع خصم 4 نقط من رخصة السياقة.

مادة 170
يُحظر قيادة المركبة في حالة سكر أو تحت تأثير الكحول بنسبة تتجاوز 0.8 غرام في اللتر في الدم.
يعاقب على هذه المخالفة بغرامة 5000 درهم مع خصم 6 نقط وتعليق رخصة السياقة لمدة لا تتجاوز سنة.

مادة 171
يُمنع تجاوز الخط الفاصل المستمر بين مسارات السير.
يعاقب على ذلك بغرامة 500 درهم مع خصم 3 نقط.

مادة 172
يُلزم كل سائق باحترام الضوء الأحمر للمرور وعدم تجاوزه.
يعاقب على مخالفة الإشارة الضوئية بغرامة 700 درهم مع خصم 4 نقط من رخصة السياقة.

مادة 173
لا يجوز لسائق مركبة للنقل العمومي للمسافرين قيادة المركبة أكثر من 9 ساعات في اليوم.
يعاقب على ذلك بغرامة 2000 درهم مع خصم 4 نقط.

مادة 174
يُمنع الانعطاف بدون الإشارة المسبقة بواسطة المؤشر الضوئي.
يعاقب على ذلك بغرامة 300 درهم مع خصم 1 نقطة.

مادة 175
يُلزم سائق الدراجة بارتداء سترة عاكسة للضوء ليلاً أو في حالة الضباب.
يعاقب على عدم الامتثال بغرامة 200 درهم.

مادة 176
يُمنع إلقاء أي شيء من المركبة أثناء السير.
يعاقب على هذا السلوك بغرامة 200 درهم.

مادة 177
يجب التوقف التام قبل الاندماج في الطريق السيار وإعطاء الأولوية للمركبات المتواجدة فيه.
يعاقب على عدم الامتثال بغرامة 700 درهم مع خصم 4 نقط.

مادة 178
يُمنع الرجوع إلى الوراء على الطريق السيار والنفق.
يعاقب على ذلك بغرامة 1500 درهم مع خصم 4 نقط.

مادة 184
يعاقب كل سائق تجاوز السرعة القانونية المحددة بأكثر من 30 كيلومترا بغرامة 700 درهم
مع خصم 4 نقط من رخصة السياقة.

مادة 185
يعاقب كل سائق قام بالوقوف في مكان ممنوع بغرامة 300 درهم
مع خصم 1 نقطة.
"""

# ================
# CORPUS FRANÇAIS 
# ================
raw_text_french = """
Article 161
La vitesse maximale à l'intérieur des agglomérations est fixée à 60 km/h pour tous les véhicules.
Tout conducteur dépassant cette vitesse est sanctionné d'une amende de 700 dirhams avec retrait de 4 points du permis de conduire.

Article 162
La vitesse maximale hors agglomérations est fixée à 100 km/h pour les véhicules légers.
Tout conducteur dépassant cette limite est sanctionné d'une amende de 700 dirhams avec retrait de 4 points.

Article 163
La vitesse maximale sur autoroute est fixée à 120 km/h.
En cas de dépassement, le conducteur est sanctionné d'une amende de 1500 dirhams avec retrait de 6 points du permis.

Article 164
Le stationnement est interdit devant les hôpitaux, les écoles, sur les ponts et dans les tunnels.
Toute infraction est sanctionnée d'une amende de 400 dirhams avec retrait de 2 points du permis de conduire.

Article 165
Le conducteur est tenu de porter la ceinture de sécurité en permanence, que ce soit en agglomération ou hors agglomération.
Le non-respect de cette obligation est sanctionné d'une amende de 300 dirhams avec retrait de 2 points.

Article 166
L'utilisation du téléphone portable en conduisant sans dispositif mains libres est interdite.
Cette infraction est sanctionnée d'une amende de 500 dirhams avec retrait de 2 points du permis.

Article 167
Les conducteurs et passagers de motocycles sont tenus de porter un casque de protection homologué.
Le non-respect est sanctionné d'une amende de 300 dirhams avec retrait de 2 points.

Article 168
La vitesse maximale pour les poids lourds hors agglomérations est fixée à 80 km/h.
Tout conducteur de poids lourd dépassant cette vitesse est sanctionné d'une amende de 700 dirhams avec retrait de 4 points.

Article 169
Tout conducteur est tenu de marquer l'arrêt obligatoire au panneau STOP et de céder la priorité.
Le non-respect est sanctionné d'une amende de 500 dirhams avec retrait de 4 points du permis.

Article 170
La conduite en état d'ivresse ou sous l'influence de l'alcool avec un taux dépassant 0,8 g/l dans le sang est interdite.
Cette infraction est sanctionnée d'une amende de 5000 dirhams, retrait de 6 points et suspension du permis pendant un an maximum.

Article 171
Le franchissement de la ligne continue séparant les voies de circulation est interdit.
Cette infraction est sanctionnée d'une amende de 500 dirhams avec retrait de 3 points.

Article 172
Tout conducteur est tenu de respecter le feu rouge et de ne pas le franchir.
Le non-respect du feu rouge est sanctionné d'une amende de 700 dirhams avec retrait de 4 points du permis.

Article 173
Le conducteur d'un véhicule de transport public de voyageurs ne peut pas conduire plus de 9 heures par jour.
Cette infraction est sanctionnée d'une amende de 2000 dirhams avec retrait de 4 points.

Article 174
Tout changement de direction sans signalisation préalable à l'aide du clignotant est interdit.
Cette infraction est sanctionnée d'une amende de 300 dirhams avec retrait de 1 point.

Article 175
Le cycliste est tenu de porter un gilet réfléchissant la nuit ou par temps de brouillard.
Le non-respect est sanctionné d'une amende de 200 dirhams.

Article 176
Il est interdit de jeter tout objet depuis un véhicule en marche.
Cette infraction est sanctionnée d'une amende de 200 dirhams.

Article 177
Tout conducteur doit s'arrêter avant d'accéder à l'autoroute et céder la priorité aux véhicules qui s'y trouvent.
Le non-respect est sanctionné d'une amende de 700 dirhams avec retrait de 4 points.

Article 178
La marche arrière sur autoroute et dans les tunnels est interdite.
Cette infraction est sanctionnée d'une amende de 1500 dirhams avec retrait de 4 points.

Article 184
Tout conducteur dépassant la vitesse légale de plus de 30 km/h encourt une amende de 700 dirhams
avec retrait de 4 points du permis de conduire.

Article 185
Tout conducteur stationnant dans un endroit interdit est sanctionné d'une amende de 300 dirhams
avec retrait de 1 point du permis de conduire.
"""

print(f"Corpus arabe    : {len(raw_text_arabic.strip().splitlines())} lignes")
print(f"Corpus français : {len(raw_text_french.strip().splitlines())} lignes")
print("\n--- Extrait arabe ---")
print(raw_text_arabic[:300])
print("\n--- Extrait français ---")
print(raw_text_french[:300])

Corpus arabe    : 79 lignes
Corpus français : 79 lignes

--- Extrait arabe ---

مادة 161
يُحدَّد الحد الأقصى للسرعة داخل التجمعات العمرانية في 60 كيلومترا في الساعة لجميع المركبات.
يعاقب كل سائق تجاوز هذه السرعة بغرامة 700 درهم مع خصم 4 نقط من رخصة السياقة.

مادة 162
يُحدَّد الحد الأقصى للسرعة خارج التجمعات العمرانية في 100 كيلومترا في الساعة للسيارات الخفيفة.
يعاقب كل سائق تج

--- Extrait français ---

Article 161
La vitesse maximale à l'intérieur des agglomérations est fixée à 60 km/h pour tous les véhicules.
Tout conducteur dépassant cette vitesse est sanctionné d'une amende de 700 dirhams avec retrait de 4 points du permis de conduire.

Article 162
La vitesse maximale hors agglomérations est f


---
## Étape A — Prétraitement Arabe (Normalisation)

Objectifs :
1. **Suppression du Tashkeel** (voyelles courtes : ً ٌ ٍ َ ُ ِ ّ ْ) → évite les biais de recherche
2. **Normalisation de la Hamza** (أ إ آ ا → ا)
3. **Normalisation de la Ta Marbuta** (ة → ه)
4. **Normalisation du Ya** (ى → ي)
5. **Suppression du Tatweel** (ـ) — caractère d'allongement
6. **Suppression des espaces multiples** et caractères non arabes parasites

In [28]:
def normalize_arabic(text: str) -> str:
    if not text:
        return text
    
    # 1. Suppression du Tashkeel via PyArabic
    if PYARABIC_AVAILABLE:
        text = strip_tashkeel(text)
        text = strip_tatweel(text)
    else:
        # Fallback manuel : plage Unicode du Tashkeel
        tashkeel = re.compile(r'[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06DC\u06DF-\u06E4\u06E7\u06E8\u06EA-\u06ED]')
        text = tashkeel.sub('', text)
        text = text.replace('ـ', '')  # Tatweel
    
    # 2. Normalisation Hamza 
    text = re.sub(r'[أإآٱ]', 'ا', text)
    
    # 3. Ta Marbuta → Heh 
    text = re.sub(r'ة', 'ه', text)
    
    # 4. Alef Maqsura → Ya
    text = re.sub(r'ى', 'ي', text)
    
    # 5. Waw avec hamza → Waw simple
    text = re.sub(r'[ؤ]', 'و', text)
    
    # 6. Ya avec hamza → Ya simple
    text = re.sub(r'[ئ]', 'ي', text)
    
    # 7. Nettoyage des espaces multiples
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    
    return text.strip()


def normalize_french(text: str) -> str:
    """Normalisation légère du texte français juridique."""
    # Mettre en minuscules
    text = text.lower()
    # Supprimer la ponctuation excessive
    text = re.sub(r'[\.,;:!?]', ' ', text)
    # Espaces multiples
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


clean_text_arabic  = normalize_arabic(raw_text_arabic)
clean_text_french  = normalize_french(raw_text_french)

print("=== Avant normalisation (arabe) ===")
print(raw_text_arabic[:200])
print("\n=== Après normalisation (arabe) ===")
print(clean_text_arabic[:200])
print("\n=== Après normalisation (français) ===")
print(clean_text_french[:200])

=== Avant normalisation (arabe) ===

مادة 161
يُحدَّد الحد الأقصى للسرعة داخل التجمعات العمرانية في 60 كيلومترا في الساعة لجميع المركبات.
يعاقب كل سائق تجاوز هذه السرعة بغرامة 700 درهم مع خصم 4 نقط من رخصة السياقة.

مادة 162
يُحدَّد الح

=== Après normalisation (arabe) ===
ماده 161
يحدد الحد الاقصي للسرعه داخل التجمعات العمرانيه في 60 كيلومترا في الساعه لجميع المركبات.
يعاقب كل سايق تجاوز هذه السرعه بغرامه 700 درهم مع خصم 4 نقط من رخصه السياقه.

ماده 162
يحدد الحد الاقص

=== Après normalisation (français) ===
article 161 la vitesse maximale à l'intérieur des agglomérations est fixée à 60 km/h pour tous les véhicules tout conducteur dépassant cette vitesse est sanctionné d'une amende de 700 dirhams avec ret


---
## Étape B — Segmentation (Tokenization)

Découper le texte en **articles individuels** puis séparer :
- L'**énoncé de l'infraction** (définition ou obligation)
- La **sanction associée** (amende + points)

In [29]:
def segment_articles_arabic(text: str) -> list:
    
    pattern = re.compile(r'(?:مادة|ماده)\s+(\d+)', re.UNICODE)
    
    parts = pattern.split(text)
    articles = []
    
    # parts = [texte_avant, num1, texte1, num2, texte2, ...]
    for i in range(1, len(parts), 2):
        article_id = parts[i].strip()
        article_body = parts[i + 1].strip() if i + 1 < len(parts) else ""
        if article_body:
            articles.append((article_id, article_body))
    
    return articles


def segment_articles_french(text: str) -> list:
    
    pattern = re.compile(r'Article\s+(\d+)', re.IGNORECASE)
    parts = pattern.split(raw_text_french)
    articles = []
    
    for i in range(1, len(parts), 2):
        article_id = parts[i].strip()
        article_body = parts[i + 1].strip() if i + 1 < len(parts) else ""
        if article_body:
            articles.append((article_id, article_body))
    
    return articles


def split_infraction_sanction_ar(text: str):
    sanction_markers = [
        r'يعاقب\s+(?:على\s+)?(?:كل\s+)?(?:سائق\s+)?(?:هذا\s+)?(?:مخالفه|هذه\s+المقتضيات|ذلك|عدم\s+الامتثال|هذا\s+السلوك|هذه\s+المخالفه|مخالفه\s+هذا\s+الالتزام|مخالفه\s+الاشاره\s+الضوئيه)?',
        r'يعاقب',
        r'يُعاقب',
    ]
    
    for marker in sanction_markers:
        m = re.search(marker, text, re.UNICODE)
        if m:
            infraction = text[:m.start()].strip()
            sanction   = text[m.start():].strip()
            return infraction if infraction else text, sanction
    
    return text, ""


def split_infraction_sanction_fr(text: str):
    sanction_markers = [
        r'(?:est|sont)\s+sanctionné',
        r'encourt\s+une\s+amende',
        r'Le\s+non-respect',
        r'Tout\s+conducteur\s+(?:de|.+?)\s+(?:est|encourt)',
    ]
    
    for marker in sanction_markers:
        m = re.search(marker, text, re.IGNORECASE)
        if m:
            infraction = text[:m.start()].strip()
            sanction   = text[m.start():].strip()
            return infraction if infraction else text, sanction
    
    return text, ""


articles_ar = segment_articles_arabic(clean_text_arabic)
articles_fr = segment_articles_french(raw_text_french)

print(f"Articles arabes segmentés   : {len(articles_ar)}")
print(f"Articles français segmentés : {len(articles_fr)}")
print("\n--- Exemple segmentation arabe ---")
for art_id, body in articles_ar[:2]:
    infr, sanct = split_infraction_sanction_ar(body)
    print(f"[Art. {art_id}]")
    print(f"  Infraction : {infr[:80]}..." if len(infr)>80 else f"  Infraction : {infr}")
    print(f"  Sanction   : {sanct[:80]}..." if len(sanct)>80 else f"  Sanction   : {sanct}")
    print()

Articles arabes segmentés   : 20
Articles français segmentés : 20

--- Exemple segmentation arabe ---
[Art. 161]
  Infraction : يحدد الحد الاقصي للسرعه داخل التجمعات العمرانيه في 60 كيلومترا في الساعه لجميع ا...
  Sanction   : يعاقب كل سايق تجاوز هذه السرعه بغرامه 700 درهم مع خصم 4 نقط من رخصه السياقه.

[Art. 162]
  Infraction : يحدد الحد الاقصي للسرعه خارج التجمعات العمرانيه في 100 كيلومترا في الساعه للسيار...
  Sanction   : يعاقب كل سايق تجاوز هذه السرعه بغرامه 700 درهم مع خصم 4 نقط.



---
## Étape C — Extraction d'Entités (NER) par Patterns Regex

In [30]:
# ============================================================
# PATTERNS REGEX — Approche Rules-based NLP
# ============================================================

PATTERN_AMENDE_AR = re.compile(
    r'(\d+)\s*(?:درهم|دراهم)',
    re.UNICODE
)

PATTERN_AMENDE_FR = re.compile(
    r'(?:amende\s+de\s+)?(\d+)\s*(?:dirhams?)',
    re.IGNORECASE
)

PATTERN_POINTS_AR = re.compile(
    r'خصم\s+(\d+)\s+نق[طه]',
    re.UNICODE
)

PATTERN_POINTS_FR = re.compile(
    r'retrait\s+de\s+(\d+)\s+points?',
    re.IGNORECASE
)

PATTERN_VITESSE_AR = re.compile(
    r'(\d+)\s*كيلومترا\s+في\s+الساعه?',
    re.UNICODE
)

PATTERN_VITESSE_FR = re.compile(
    r'(\d+)\s*km/h',
    re.IGNORECASE
)

PATTERN_SUSPENSION_AR = re.compile(
    r'تعليق\s+رخصه\s+السياقه?(?:\s+لمده\s+لا\s+تتجاوز\s+(\w+))?',
    re.UNICODE
)

PATTERN_SUSPENSION_FR = re.compile(
    r'suspension\s+du\s+permis(?:\s+pendant\s+([\w\s]+))?',
    re.IGNORECASE
)

PATTERN_ROUTE_FR = re.compile(
    r'(?:route|autoroute|voie)\s+(?:n°|numéro\s+)?(\d+)',
    re.IGNORECASE
)

PATTERN_ALCOOL = re.compile(
    r'(\d+[,.]?\d*)\s*g(?:rammes?)?(?:/l|\s+par\s+litre|\s+في\s+اللتر)',
    re.IGNORECASE | re.UNICODE
)

PATTERN_DUREE = re.compile(
    r'(\d+)\s*(?:heures?|ساعات?|ساعه?)(?:\s+(?:par|في)\s+(?:jour|اليوم))?',
    re.IGNORECASE | re.UNICODE
)


# ============================================================
# DICTIONNAIRES DE CLASSIFICATION
# ============================================================

# Catégorie véhicule
VEHICULE_MAP = {
    'poids_lourd': [
        'شاحنه', 'شاحنة', 'مركبه.*ثقيله', 'poids lourd', 'véhicule.*lourd',
        'transport.*marchandises', 'البضائع'
    ],
    'transport_commun': [
        'نقل.*عمومي', 'نقل.*مسافرين', 'transport.*public', 'voyageurs'
    ],
    'moto': [
        'دراجه.*ناريه', 'دراجة نارية', 'motocycles?', 'moto(?:cyclette)?'
    ],
    'velo': [
        'دراجه(?!.*ناريه)', 'cycliste', 'vélo', 'bicyclette'
    ],
    'tous_vehicules': [
        'جميع\s+المركبات', 'tous\s+les\s+véhicules'
    ]
}

# Localisation
LOCALISATION_MAP = {
    'autoroute': [
        'الطريق\s+السيار', 'autoroute'
    ],
    'agglomeration': [
        'التجمعات\s+العمراني', 'داخل\s+التجمع', 'en\s+agglomération', 'à\s+l.intérieur'
    ],
    'hors_agglomeration': [
        'خارج\s+التجمع', 'hors\s+agglomération'
    ],
    'tunnel': [
        'النفق', 'tunnel'
    ],
    'pont': [
        'الجسور', 'ponts?'
    ],
    'intersection': [
        'التوقف\s+\(STOP\)', 'panneau\s+STOP', 'carrefour'
    ]
}

# Type d'infraction
TYPE_INFRACTION_MAP = {
    'vitesse': [
        'السرعه', 'السرعة', 'vitesse', 'km/h', 'كيلومترا'
    ],
    'stationnement': [
        'الوقوف', 'الوقوف\s+في\s+مكان', 'stationnement', 'stationner', 'parking'
    ],
    'alcool': [
        'سكر', 'كحول', 'alcool', 'ivresse'
    ],
    'telephone': [
        'هاتف\s+نقال', 'téléphone', 'portable'
    ],
    'ceinture': [
        'حزام\s+الامان', 'ceinture'
    ],
    'casque': [
        'خوذه', 'casque'
    ],
    'feu_rouge': [
        'الضوء\s+الاحمر', 'الاشاره\s+الضوئيه', 'feu\s+rouge'
    ],
    'depassement': [
        'الخط\s+الفاصل', 'franchissement.*ligne', 'ligne\s+continue'
    ],
    'priorite': [
        'التوقف.*STOP', 'الاولويه', 'priorité', 'STOP'
    ],
    'fatigue_conducteur': [
        'ساعات.*يوم', '9\s+ساعات', 'heures.*jour', 'durée.*conduite'
    ],
    'clignotant': [
        'الانعطاف', 'مؤشر.*ضوئي', 'clignotant', 'changement.*direction'
    ],
    'gilet_reflechissant': [
        'سترة\s+عاكسه', 'سترة عاكسة', 'gilet.*réfléchissant'
    ],
    'jet_ordures': [
        'إلقاء', 'jeter.*objet'
    ],
    'marche_arriere': [
        'الرجوع.*الوراء', 'marche\s+arrière'
    ]
}

def get_gravite(amende: float, points: int, suspension: bool) -> str:
    if suspension or amende >= 3000 or points >= 6:
        return 'très_grave'
    elif amende >= 700 or points >= 4:
        return 'grave'
    elif amende >= 300 or points >= 2:
        return 'modérée'
    else:
        return 'légère'


def match_category(text: str, category_map: dict) -> str:
    text_lower = text.lower()
    for category, patterns in category_map.items():
        for p in patterns:
            if re.search(p, text_lower, re.IGNORECASE | re.UNICODE):
                return category
    return 'autre'


def extract_mots_cles(text_ar: str, text_fr: str) -> list:
    mots_cles = []
    
    # Mots-clés arabes
    kw_ar = {
        'vitesse': ['السرعه', 'كيلومترا'],
        'nuit': ['ليلا', 'ليلاً'],
        'autoroute': ['الطريق\s+السيار'],
        'alcool': ['سكر', 'كحول'],
        'permis': ['رخصه\s+السياقه'],
        'agglomeration': ['التجمعات\s+العمراني'],
        'stationnement': ['الوقوف'],
        'stop': ['STOP', 'الاولويه'],
        'telephone': ['هاتف\s+نقال'],
    }
    
    for kw, patterns in kw_ar.items():
        for p in patterns:
            if re.search(p, text_ar, re.UNICODE):
                if kw not in mots_cles:
                    mots_cles.append(kw)
    
    # Mots-clés français
    kw_fr = {
        'nuit': ['nuit', 'brouillard'],
        'tunnel': ['tunnel'],
        'pont': ['pont'],
        'mains_libres': ['mains libres'],
        'homologue': ['homologué'],
    }
    
    for kw, patterns in kw_fr.items():
        for p in patterns:
            if re.search(p, text_fr, re.IGNORECASE):
                if kw not in mots_cles:
                    mots_cles.append(kw)
    
    return mots_cles


print("✔ Patterns et dictionnaires chargés")
print(f"  Catégories véhicule    : {list(VEHICULE_MAP.keys())}")
print(f"  Catégories localisation: {list(LOCALISATION_MAP.keys())}")
print(f"  Types d'infraction     : {list(TYPE_INFRACTION_MAP.keys())}")

✔ Patterns et dictionnaires chargés
  Catégories véhicule    : ['poids_lourd', 'transport_commun', 'moto', 'velo', 'tous_vehicules']
  Catégories localisation: ['autoroute', 'agglomeration', 'hors_agglomeration', 'tunnel', 'pont', 'intersection']
  Types d'infraction     : ['vitesse', 'stationnement', 'alcool', 'telephone', 'ceinture', 'casque', 'feu_rouge', 'depassement', 'priorite', 'fatigue_conducteur', 'clignotant', 'gilet_reflechissant', 'jet_ordures', 'marche_arriere']


In [31]:
def extract_article_data(article_id: str, text_ar: str, text_fr: str) -> dict:
   
    text_ar_norm = normalize_arabic(text_ar)
    
    infr_ar, sanct_ar = split_infraction_sanction_ar(text_ar_norm)
    infr_fr, sanct_fr = split_infraction_sanction_fr(text_fr)
    
    amende = None
    m = PATTERN_AMENDE_AR.search(text_ar_norm)
    if m:
        amende = int(m.group(1))
    elif (m2 := PATTERN_AMENDE_FR.search(text_fr)):
        amende = int(m2.group(1))
    
    points = None
    m = PATTERN_POINTS_AR.search(text_ar_norm)
    if m:
        points = int(m.group(1))
    elif (m2 := PATTERN_POINTS_FR.search(text_fr)):
        points = int(m2.group(1))
    
    vitesse = None
    m = PATTERN_VITESSE_AR.search(text_ar_norm)
    if m:
        vitesse = int(m.group(1))
    elif (m2 := PATTERN_VITESSE_FR.search(text_fr)):
        vitesse = int(m2.group(1))
    
    suspension = bool(
        PATTERN_SUSPENSION_AR.search(text_ar_norm) or
        PATTERN_SUSPENSION_FR.search(text_fr)
    )
    
    taux_alcool = None
    m = PATTERN_ALCOOL.search(text_ar_norm + " " + text_fr)
    if m:
        taux_alcool = float(m.group(1).replace(',', '.'))
    
    duree_max = None
    m = PATTERN_DUREE.search(text_ar_norm + " " + text_fr)
    if m and ('يوم' in text_ar_norm or 'jour' in text_fr.lower()):
        duree_max = int(m.group(1))
    
    combined = text_ar_norm + " " + text_fr
    categorie_vehicule = match_category(combined, VEHICULE_MAP)
    localisation       = match_category(combined, LOCALISATION_MAP)
    type_infraction    = match_category(combined, TYPE_INFRACTION_MAP)
    
    role = classify_role_regles(text_ar_norm, text_fr)
    
    mots_cles = extract_mots_cles(text_ar_norm, text_fr)
    
    gravite = get_gravite(
        amende or 0, 
        points or 0, 
        suspension
    )
    
    desc = infr_fr.strip().replace('\n', ' ').strip()
    if not desc:
        desc = infr_ar.strip().replace('\n', ' ').strip()
    
    return {
        'article_id'        : article_id,
        'infraction_desc_fr': infr_fr.replace('\n', ' ').strip(),
        'infraction_desc_ar': infr_ar.replace('\n', ' ').strip(),
        'sanction_desc_fr'  : sanct_fr.replace('\n', ' ').strip(),
        'amende_fixe'       : amende,
        'points_retrait'    : points,
        'vitesse_limite'    : vitesse,
        'suspension_permis' : suspension,
        'taux_alcool_g_l'   : taux_alcool,
        'duree_conduite_max': duree_max,
        'categorie_vehicule': categorie_vehicule,
        'localisation'      : localisation,
        'type_infraction'   : type_infraction,
        'role_paragraphe'   : role,
        'niveau_gravite'    : gravite,
        'mots_cles'         : '|'.join(mots_cles),
    }


def classify_role_regles(text_ar: str, text_fr: str) -> str:
    
    if re.search(r'يعاقب|يُعاقب|غرامه|خصم\s+\d', text_ar, re.UNICODE) or \
       re.search(r'sanctionné|amende|retrait\s+de\s+\d', text_fr, re.IGNORECASE):
        return 'sanction'
    
   
    elif re.search(r'يُلزم|يجب|يُحظر|يُمنع|ملزم', text_ar, re.UNICODE) or \
         re.search(r'est\s+tenu|est\s+interdit|est\s+obligatoire|doit|interdite?', text_fr, re.IGNORECASE):
        return 'obligation'
    
    else:
        return 'définition'


print("✔ Fonctions d'extraction définies")

✔ Fonctions d'extraction définies


---
## Construction du DataFrame principal

In [32]:

articles_ar = segment_articles_arabic(clean_text_arabic)
articles_fr = segment_articles_french(raw_text_french)

fr_dict = {art_id: body for art_id, body in articles_fr}

rows = []
for art_id, body_ar in articles_ar:
    body_fr = fr_dict.get(art_id, "")
    row = extract_article_data(art_id, body_ar, body_fr)
    rows.append(row)

df = pd.DataFrame(rows)

df['article_id']   = df['article_id'].astype(int)
df['amende_fixe']  = pd.to_numeric(df['amende_fixe'], errors='coerce')
df['points_retrait'] = pd.to_numeric(df['points_retrait'], errors='coerce')
df['vitesse_limite'] = pd.to_numeric(df['vitesse_limite'], errors='coerce')

df = df.sort_values('article_id').reset_index(drop=True)

print(f"\n-> DataFrame construit : {len(df)} articles, {len(df.columns)} colonnes")
print("\nColonnes :", list(df.columns))
print()
df[['article_id', 'type_infraction', 'amende_fixe', 'points_retrait', 'niveau_gravite', 'categorie_vehicule']]


-> DataFrame construit : 20 articles, 16 colonnes

Colonnes : ['article_id', 'infraction_desc_fr', 'infraction_desc_ar', 'sanction_desc_fr', 'amende_fixe', 'points_retrait', 'vitesse_limite', 'suspension_permis', 'taux_alcool_g_l', 'duree_conduite_max', 'categorie_vehicule', 'localisation', 'type_infraction', 'role_paragraphe', 'niveau_gravite', 'mots_cles']



,article_id,type_infraction,amende_fixe,points_retrait,niveau_gravite,categorie_vehicule
0,161,vitesse,700,4.0,grave,tous_vehicules
1,162,vitesse,700,4.0,grave,autre
2,163,vitesse,1500,6.0,très_grave,autre
3,164,stationnement,400,2.0,modérée,autre
4,165,ceinture,300,2.0,modérée,autre
5,166,telephone,500,2.0,modérée,autre
6,167,casque,300,2.0,modérée,moto
7,168,vitesse,700,4.0,grave,poids_lourd
8,169,priorite,500,4.0,grave,autre
9,170,alcool,5000,6.0,très_grave,autre


In [33]:
# Affichage complet du DataFrame
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_columns', None)
df

,article_id,infraction_desc_fr,infraction_desc_ar,sanction_desc_fr,amende_fixe,points_retrait,vitesse_limite,suspension_permis,taux_alcool_g_l,duree_conduite_max,categorie_vehicule,localisation,type_infraction,role_paragraphe,niveau_gravite,mots_cles
0,161,La vitesse maximale à l'intérieur des agglomérations est...,يحدد الحد الاقصي للسرعه داخل التجمعات العمرانيه في 60 كي...,est sanctionné d'une amende de 700 dirhams avec retrait ...,700,4.0,60.0,False,NaN,NaN,tous_vehicules,agglomeration,vitesse,sanction,grave,vitesse|permis|agglomeration
1,162,La vitesse maximale hors agglomérations est fixée à 100 ...,يحدد الحد الاقصي للسرعه خارج التجمعات العمرانيه في 100 ك...,est sanctionné d'une amende de 700 dirhams avec retrait ...,700,4.0,100.0,False,NaN,NaN,autre,agglomeration,vitesse,sanction,grave,vitesse|agglomeration
2,163,La vitesse maximale sur autoroute est fixée à 120 km/h. ...,يحدد الحد الاقصي للسرعه علي الطريق السيار في 120 كيلومتر...,est sanctionné d'une amende de 1500 dirhams avec retrait...,1500,6.0,120.0,False,NaN,NaN,autre,autoroute,vitesse,sanction,très_grave,vitesse|autoroute|permis
3,164,"Le stationnement est interdit devant les hôpitaux, les é...",يمنع الوقوف في الاماكن الاتيه: امام المستشفيات، امام الم...,est sanctionnée d'une amende de 400 dirhams avec retrait...,400,2.0,NaN,False,NaN,NaN,autre,tunnel,stationnement,sanction,modérée,permis|stationnement|tunnel|pont
4,165,Le conducteur est tenu de porter la ceinture de sécurité...,يجب علي السايق ربط حزام الامان طوال مده السياقه داخل وخا...,est sanctionné d'une amende de 300 dirhams avec retrait ...,300,2.0,NaN,False,NaN,NaN,autre,agglomeration,ceinture,sanction,modérée,agglomeration
5,166,L'utilisation du téléphone portable en conduisant sans d...,يمنع استعمال الهاتف النقال اثناء قياده المركبه دون استعم...,est sanctionnée d'une amende de 500 dirhams avec retrait...,500,2.0,NaN,False,NaN,NaN,autre,autre,telephone,sanction,modérée,permis|mains_libres
6,167,Les conducteurs et passagers de motocycles sont tenus de...,يلزم سايقو الدراجات الناريه وركابها بارتداء خوذه واقيه م...,est sanctionné d'une amende de 300 dirhams avec retrait ...,300,2.0,NaN,False,NaN,NaN,moto,autre,casque,sanction,modérée,homologue
7,168,La vitesse maximale pour les poids lourds hors aggloméra...,يحدد الحد الاقصي للسرعه لمركبات نقل البضايع الثقيله خارج...,est sanctionné d'une amende de 700 dirhams avec retrait ...,700,4.0,80.0,False,NaN,NaN,poids_lourd,agglomeration,vitesse,sanction,grave,vitesse|agglomeration
8,169,Tout conducteur est tenu de marquer l'arrêt obligatoire ...,يلزم كل سايق بالتوقف عند اشاره التوقف (STOP) واعطاء الاو...,est sanctionné d'une amende de 500 dirhams avec retrait ...,500,4.0,NaN,False,NaN,NaN,autre,intersection,priorite,sanction,grave,permis|stop
9,170,La conduite en état d'ivresse ou sous l'influence de l'a...,يحظر قياده المركبه في حاله سكر او تحت تاثير الكحول بنسبه...,"est sanctionnée d'une amende de 5000 dirhams, retrait de...",5000,6.0,NaN,True,0.8,NaN,autre,autre,alcool,sanction,très_grave,alcool|permis


---
## Étape D — Vectorisation & Classification ML (TF-IDF + KMeans)

Regroupement automatique des infractions similaires par clustering non supervisé.

In [34]:
# ============================================================
# APPROCHE 1 : TF-IDF + KMeans (Machine Learning)
# ============================================================

corpus = [
    row['infraction_desc_fr'] + " " + row['infraction_desc_ar']
    for _, row in df.iterrows()
]

# TF-IDF Vectorisation
tfidf = TfidfVectorizer(
    analyzer='char_wb',    # N-grammes de caractères (supporte l'arabe)
    ngram_range=(2, 4),
    max_features=500,
    min_df=1,
    sublinear_tf=True
)

X = tfidf.fit_transform(corpus)
print(f"Matrice TF-IDF : {X.shape[0]} docs × {X.shape[1]} features")

inertias = []
K_range = range(2, min(8, len(df)))

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X)
    inertias.append(km.inertia_)

print("\nInertie par nombre de clusters :")
for k, inertia in zip(K_range, inertias):
    print(f"  k={k} → inertie={inertia:.2f}")

N_CLUSTERS = 5
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init=10)
df['cluster_tfidf'] = kmeans.fit_predict(X)

cluster_names = {}
for cluster_id in range(N_CLUSTERS):
    mask = df['cluster_tfidf'] == cluster_id
    if mask.any():
        dominant = df[mask]['type_infraction'].mode()[0]
        cluster_names[cluster_id] = f"cluster_{cluster_id}_{dominant}"
    else:
        cluster_names[cluster_id] = f"cluster_{cluster_id}"

df['cluster_nom'] = df['cluster_tfidf'].map(cluster_names)

print(f"\n-> KMeans appliqué avec {N_CLUSTERS} clusters")
print("\nRépartition par cluster :")
print(df.groupby('cluster_nom')['article_id'].apply(list).to_string())

Matrice TF-IDF : 20 docs × 500 features

Inertie par nombre de clusters :
  k=2 → inertie=9.05
  k=3 → inertie=7.07
  k=4 → inertie=6.33
  k=5 → inertie=5.73
  k=6 → inertie=5.00
  k=7 → inertie=4.58

-> KMeans appliqué avec 5 clusters

Répartition par cluster :
cluster_nom
cluster_0_priorite         [165, 167, 169, 172, 175, 177]
cluster_1_vitesse               [161, 162, 163, 168, 184]
cluster_2_alcool                [166, 170, 173, 176, 178]
cluster_3_clignotant                      [164, 171, 174]
cluster_4_stationnement                             [185]


---
## Étape E — Modèle Pré-entraîné : NER avec spaCy + NER Custom

Utilisation de spaCy pour l'extraction d'entités nommées sur le texte français.  
En l'absence du modèle `fr_core_news_sm`, un **NER custom basé sur règles** est utilisé comme fallback robuste.

In [35]:
# ============================================================
# APPROCHE 2 : NER avec spaCy (modèle pré-entraîné)
# ============================================================

def ner_spacy(text: str) -> list:
    if not SPACY_AVAILABLE:
        return []
    doc = nlp_fr(text)
    return [(ent.text, ent.label_) for ent in doc.ents]


def ner_custom_juridique(text: str) -> list:
    entities = []
    
    # AMENDE
    for m in re.finditer(r'(\d+)\s*(?:درهم|دراهم|dirhams?)', text, re.IGNORECASE | re.UNICODE):
        entities.append((m.group(0).strip(), 'AMENDE'))
    
    # POINTS
    for m in re.finditer(r'(?:خصم\s+)?(\d+)\s+(?:نق[طه]|points?)', text, re.IGNORECASE | re.UNICODE):
        entities.append((m.group(0).strip(), 'POINTS_RETRAIT'))
    
    # VITESSE
    for m in re.finditer(r'(\d+)\s*(?:كيلومترا\s+في\s+الساعه?|km/h)', text, re.IGNORECASE | re.UNICODE):
        entities.append((m.group(0).strip(), 'VITESSE'))
    
    # NUMÉRO ARTICLE
    for m in re.finditer(r'(?:مادة|Article)\s+(\d+)', text, re.UNICODE):
        entities.append((m.group(0).strip(), 'ARTICLE'))
    
    # TAUX ALCOOL
    for m in re.finditer(r'(\d+[,.]\d+)\s*g(?:/l|\s*par litre|\s+في اللتر)', text, re.IGNORECASE | re.UNICODE):
        entities.append((m.group(0).strip(), 'TAUX_ALCOOL'))
    
    # DURÉE
    for m in re.finditer(r'(\d+)\s*(?:heures?|ساعات?)', text, re.IGNORECASE | re.UNICODE):
        entities.append((m.group(0).strip(), 'DURÉE'))
    
    # SUSPENSION
    for m in re.finditer(r'(?:تعليق رخصه|suspension du permis)(?:\s+\w+){0,5}', text, re.IGNORECASE | re.UNICODE):
        entities.append((m.group(0).strip(), 'SUSPENSION'))
    
    return entities


# Appliquer NER sur chaque article
ner_results = []
for _, row in df.iterrows():
    combined = row['infraction_desc_fr'] + ' ' + row['sanction_desc_fr'] + ' ' + row['infraction_desc_ar']
    
    # Essayer spaCy d'abord, sinon NER custom
    if SPACY_AVAILABLE:
        entities = ner_spacy(row['infraction_desc_fr'])
        entities += ner_custom_juridique(combined)  # Compléter avec custom
    else:
        entities = ner_custom_juridique(combined)
    
    ner_results.append(entities)

df['entites_ner'] = ner_results

# Affichage des entités extraites
print("=== Entités NER extraites (custom) ===")
for _, row in df.head(5).iterrows():
    print(f"\nArticle {row['article_id']} ({row['type_infraction']}) :")
    for ent, label in row['entites_ner']:
        print(f"  [{label:20s}] → {ent}")

=== Entités NER extraites (custom) ===

Article 161 (vitesse) :
  [AMENDE              ] → 700 dirhams
  [POINTS_RETRAIT      ] → 4 points
  [VITESSE             ] → 60 km/h
  [VITESSE             ] → 60 كيلومترا في الساعه

Article 162 (vitesse) :
  [AMENDE              ] → 700 dirhams
  [POINTS_RETRAIT      ] → 4 points
  [VITESSE             ] → 100 km/h
  [VITESSE             ] → 100 كيلومترا في الساعه

Article 163 (vitesse) :
  [MISC                ] → En cas de dépassement
  [AMENDE              ] → 1500 dirhams
  [POINTS_RETRAIT      ] → 6 points
  [VITESSE             ] → 120 km/h
  [VITESSE             ] → 120 كيلومترا في الساعه

Article 164 (stationnement) :
  [AMENDE              ] → 400 dirhams
  [POINTS_RETRAIT      ] → 2 points

Article 165 (ceinture) :
  [AMENDE              ] → 300 dirhams
  [POINTS_RETRAIT      ] → 2 points


---
## Comparaison : Approche Regex vs Modèle pré-entraîné

In [36]:
# ============================================================
# COMPARAISON : Regex (Rules-based) vs NER (Modèle ML/pré-entraîné)
# ============================================================

print("=" * 70)
print("COMPARAISON APPROCHE REGEX vs NER (spaCy/Custom)")
print("=" * 70)

comparison_data = {
    'Critère': [
        'Précision sur données connues',
        'Généralisation (textes nouveaux)',
        'Vitesse d\'exécution',
        'Besoin de données d\'entraînement',
        'Interprétabilité',
        'Maintenance',
        'Gestion de l\'ambiguïté',
        'Support multi-langue (AR + FR)',
        'Détection des entités numériques',
        'Détection des entités contextuelles',
    ],
    'Regex (Rules-based)': [
        'très haute',
        'faible si texte varie',
        'très rapide',
        'aucun',
        'totale',
        'mise à jour manuelle',
        'limité',
        'patterns séparés',
        'précis',
        'limité au vocabulaire',
    ],
    'NER (spaCy/AraBERT)': [
        'haute si bon modèle',
        'bonne généralisation',
        'plus lent',
        'nécessite données annotées',
        'boîte noir',
        'réentraînemen)',
        'gère le contexte',
        'modèles séparés',
        'moins précis',
        'excellent',
    ]
}

df_comparison = pd.DataFrame(comparison_data)
print(df_comparison.to_string(index=False))




COMPARAISON APPROCHE REGEX vs NER (spaCy/Custom)
                            Critère   Regex (Rules-based)        NER (spaCy/AraBERT)
      Précision sur données connues            très haute        haute si bon modèle
   Généralisation (textes nouveaux) faible si texte varie       bonne généralisation
                Vitesse d'exécution           très rapide                  plus lent
   Besoin de données d'entraînement                 aucun nécessite données annotées
                   Interprétabilité                totale                 boîte noir
                        Maintenance  mise à jour manuelle             réentraînemen)
             Gestion de l'ambiguïté                limité           gère le contexte
     Support multi-langue (AR + FR)      patterns séparés            modèles séparés
   Détection des entités numériques                précis               moins précis
Détection des entités contextuelles limité au vocabulaire                  excellent


---
## Statistiques et Analyse du Dataset

In [37]:
print("=" * 50)
print("STATISTIQUES DU CORPUS")
print("=" * 50)

print(f"\n- Nombre total d'articles : {len(df)}")
print(f"- Articles avec amende    : {df['amende_fixe'].notna().sum()}")
print(f"- Articles avec points    : {df['points_retrait'].notna().sum()}")
print(f"- Articles avec suspension: {df['suspension_permis'].sum()}")

print("\n--- Distribution des amendes ---")
print(df['amende_fixe'].describe())

print("\n--- Distribution par type d'infraction ---")
print(df['type_infraction'].value_counts())

print("\n--- Distribution par niveau de gravité ---")
print(df['niveau_gravite'].value_counts())

print("\n--- Distribution par catégorie de véhicule ---")
print(df['categorie_vehicule'].value_counts())

print("\n--- Top 5 amendes les plus élevées ---")
top5 = df.nlargest(5, 'amende_fixe')[['article_id', 'type_infraction', 'amende_fixe', 'points_retrait', 'niveau_gravite']]
print(top5.to_string(index=False))

STATISTIQUES DU CORPUS

- Nombre total d'articles : 20
- Articles avec amende    : 20
- Articles avec points    : 18
- Articles avec suspension: 1

--- Distribution des amendes ---
count      20.000000
mean      885.000000
std      1079.120012
min       200.000000
25%       300.000000
50%       600.000000
75%       700.000000
max      5000.000000
Name: amende_fixe, dtype: float64

--- Distribution par type d'infraction ---
type_infraction
vitesse                5
stationnement          2
priorite               2
ceinture               1
telephone              1
casque                 1
alcool                 1
depassement            1
feu_rouge              1
fatigue_conducteur     1
clignotant             1
gilet_reflechissant    1
jet_ordures            1
marche_arriere         1
Name: count, dtype: int64

--- Distribution par niveau de gravité ---
niveau_gravite
grave         9
modérée       7
très_grave    2
légère        2
Name: count, dtype: int64

--- Distribution par catégorie 

---
## Export Final — export_final.csv

In [38]:
# ============================================================
# EXPORT FINAL
# ============================================================

colonnes_csv = [
    'article_id',
    'infraction_desc_fr',
    'infraction_desc_ar',
    'sanction_desc_fr',
    'categorie_vehicule',
    'amende_fixe',
    'points_retrait',
    'vitesse_limite',
    'suspension_permis',
    'taux_alcool_g_l',
    'duree_conduite_max',
    'type_infraction',
    'localisation',
    'role_paragraphe',
    'niveau_gravite',
    'mots_cles',
    'cluster_tfidf',
    'cluster_nom',
]

df_final = df[colonnes_csv].copy()

# Export CSV encodé UTF-8-sig (compatible Excel + Windows)
df_final.to_csv('export_final.csv', index=False, encoding='utf-8-sig')

print("✅ Fichier export_final.csv généré avec succès !")
print(f"   Lignes   : {len(df_final)}")
print(f"   Colonnes : {len(df_final.columns)}")
print()
print("Colonnes exportées :")
for i, col in enumerate(df_final.columns, 1):
    print(f"  {i:2d}. {col}")

print()
print("--- Aperçu du CSV (5 premières lignes) ---")
df_final.head(5)

✅ Fichier export_final.csv généré avec succès !
   Lignes   : 20
   Colonnes : 18

Colonnes exportées :
   1. article_id
   2. infraction_desc_fr
   3. infraction_desc_ar
   4. sanction_desc_fr
   5. categorie_vehicule
   6. amende_fixe
   7. points_retrait
   8. vitesse_limite
   9. suspension_permis
  10. taux_alcool_g_l
  11. duree_conduite_max
  12. type_infraction
  13. localisation
  14. role_paragraphe
  15. niveau_gravite
  16. mots_cles
  17. cluster_tfidf
  18. cluster_nom

--- Aperçu du CSV (5 premières lignes) ---


,article_id,infraction_desc_fr,infraction_desc_ar,sanction_desc_fr,categorie_vehicule,amende_fixe,points_retrait,vitesse_limite,suspension_permis,taux_alcool_g_l,duree_conduite_max,type_infraction,localisation,role_paragraphe,niveau_gravite,mots_cles,cluster_tfidf,cluster_nom
0,161,La vitesse maximale à l'intérieur des agglomérations est...,يحدد الحد الاقصي للسرعه داخل التجمعات العمرانيه في 60 كي...,est sanctionné d'une amende de 700 dirhams avec retrait ...,tous_vehicules,700,4.0,60.0,False,NaN,NaN,vitesse,agglomeration,sanction,grave,vitesse|permis|agglomeration,1,cluster_1_vitesse
1,162,La vitesse maximale hors agglomérations est fixée à 100 ...,يحدد الحد الاقصي للسرعه خارج التجمعات العمرانيه في 100 ك...,est sanctionné d'une amende de 700 dirhams avec retrait ...,autre,700,4.0,100.0,False,NaN,NaN,vitesse,agglomeration,sanction,grave,vitesse|agglomeration,1,cluster_1_vitesse
2,163,La vitesse maximale sur autoroute est fixée à 120 km/h. ...,يحدد الحد الاقصي للسرعه علي الطريق السيار في 120 كيلومتر...,est sanctionné d'une amende de 1500 dirhams avec retrait...,autre,1500,6.0,120.0,False,NaN,NaN,vitesse,autoroute,sanction,très_grave,vitesse|autoroute|permis,1,cluster_1_vitesse
3,164,"Le stationnement est interdit devant les hôpitaux, les é...",يمنع الوقوف في الاماكن الاتيه: امام المستشفيات، امام الم...,est sanctionnée d'une amende de 400 dirhams avec retrait...,autre,400,2.0,NaN,False,NaN,NaN,stationnement,tunnel,sanction,modérée,permis|stationnement|tunnel|pont,3,cluster_3_clignotant
4,165,Le conducteur est tenu de porter la ceinture de sécurité...,يجب علي السايق ربط حزام الامان طوال مده السياقه داخل وخا...,est sanctionné d'une amende de 300 dirhams avec retrait ...,autre,300,2.0,NaN,False,NaN,NaN,ceinture,agglomeration,sanction,modérée,agglomeration,0,cluster_0_priorite
